In [1]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")
df_exp4a = pd.read_csv("data/exp4/exp4_metabolites_best_reps.csv")
df_exp4b = pd.read_csv("data/exp4/exp4_metabolites_new_best.csv")
df_exp4c = pd.read_csv("data/exp4/exp4_metabolites_new_worst.csv")
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3, df_exp4a, df_exp4b, df_exp4c))

# define variable names
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

# average replicates
df_fmt_mean = []
for exp_name, df_exp in df.groupby("Experiments"):
    df_groups = df_exp.groupby("Time")
    df_avg = df_groups[system_variables].mean().reset_index()
    df_avg.insert(0, "Experiments", [exp_name]*df_avg.shape[0])
    df_fmt_mean.append(df_avg)
df = pd.concat(df_fmt_mean)

In [3]:
# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
data = format_data(df.copy(), species, metabolites, controls, observed=observed)
data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), n_metabolites=len(metabolites), n_controls=len(controls), n_hidden=32)
# fit model
brnn.fit(data_scaled, evd_tol=1e-3)

Total measurements: 27244, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1353.323, Residuals: -0.00492
Loss: 1268.678, Residuals: -0.00160
Loss: 1238.723, Residuals: 0.00099
Loss: 1209.082, Residuals: 0.00157
Loss: 1096.335, Residuals: 0.00430
Loss: 1069.086, Residuals: -0.00229
Loss: 1020.533, Residuals: -0.00056
Loss: 936.476, Residuals: -0.00055
Loss: 901.321, Residuals: -0.00080
Loss: 839.931, Residuals: -0.00060
Loss: 799.638, Residuals: -0.00092
Loss: 791.830, Residuals: 0.00123
Loss: 727.067, Residuals: -0.00194
Loss: 720.925, Residuals: -0.00036
Loss: 672.729, Residuals: -0.00177
Loss: 655.239, Residuals: -0.00232
Loss: 623.445, Residuals: -0.00166
Loss: 590.366, Residuals: -0.00046
Loss: 584.677, Residuals: -0.00046
Loss: 579.334, Residuals: -0.00075
Loss: 569.155, Residuals: -0.00075
Loss: 554.182, Residuals: -0.00098
Loss: 533.623, Residuals: -0.00169
Loss: 523.967, Residuals: -0.00034
Loss: 522.359, Residuals: -0.00101
Loss: 508.948, Residuals: -0.00102

In [5]:
# log that ignores zeros
def zlog(x):
    return jnp.where(x > 0, jnp.log(x), 0.)

# shannon diversity
def shannon(y):
    y_rel = y / jnp.sum(y)
    return jnp.nansum(-zlog(y_rel)*y_rel)

# determine best previously observed values of objectives
b_max = 24.206144013
d_max = 2.256914760709628 
v_max = 0.05319052969911115

# define objective 
def objective(y):
    
    # endpoint shannon diversity
    diversity = shannon(y[-1, :len(species)]) 
    
    # variance in species abundances in last two passages
    species_stdv = jnp.std(y[-2:, :len(species)], 0)
    instability  = jnp.where(species_stdv>0, species_stdv, 0).mean() 
    
    # endpoint butyrate production 
    butyrate = y[-1, -2] 
    
    # scaled objective 
    scaled_obj = diversity / d_max - instability / v_max + butyrate / b_max
    
    return diversity, instability, butyrate, scaled_obj

# function handle evaluates objective in batches
objective_batch = vmap(objective)

In [6]:
# save predictions to dataframe
pred_dfs = []
for T, X, U, Y, exp_names in data_scaled: 
    
    # set Y as a copy
    Y = np.copy(Y)
    
    # keep initial condition
    P = np.array(brnn.predict_point(X, U))
    
    # unscale measurements
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        Y[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        Y[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        P[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        P[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = P.shape
    
    # measured objective
    div_meas, inst_meas, but_meas, obj_meas = objective_batch(Y)
    div_pred, inst_pred, but_pred, obj_pred = objective_batch(P)
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(Y, [batch_size*n_time, n_out])
    P = np.reshape(P, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # names of variables
    outputs = species+metabolites
    outputs_meas = [o + " meas" for o in outputs]
    outputs_pred = [o + " pred" for o in outputs]
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    
    pred_df['Diversity meas'] = np.repeat(div_meas, n_time)
    pred_df['Instability meas'] = np.repeat(inst_meas, n_time)
    pred_df['Butyrate meas'] = np.repeat(but_meas, n_time)
    pred_df['Objective meas'] = np.repeat(obj_meas, n_time)
    
    pred_df['Diversity pred'] = np.repeat(div_pred, n_time)
    pred_df['Instability pred'] = np.repeat(inst_pred, n_time)
    pred_df['Butyrate pred'] = np.repeat(but_pred, n_time)
    pred_df['Objective pred'] = np.repeat(obj_pred, n_time)
    
    pred_df[outputs_meas] = Y
    pred_df[outputs_pred] = P
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)
pred_df.to_csv("space/sampled_space.csv", index=False)

In [7]:
pred_df

,Experiments,Time,Diversity meas,Instability meas,Butyrate meas,Objective meas,Diversity pred,Instability pred,Butyrate pred,Objective pred,...,RIabs pred,pH pred,Lactate pred,Acetate pred,AcGum,ArGal,Inulin,Pectin,Starch,Xylan
0,AC-AcGum,0.0,0.0,0.000473,0.000000,NaN,0.0,0.002094,0.000000,-0.039376,...,0.0,6.700000,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,0.0
1,AC-AcGum,1.0,0.0,0.000473,NaN,NaN,0.0,0.002094,0.000000,-0.039376,...,0.0,6.764289,0.000000,4.281863,1.0,0.0,0.0,0.0,0.0,0.0
2,AC-AcGum-ArGal-Inulin-Pectin-Starch,0.0,0.0,0.003655,0.000000,NaN,0.0,0.004083,0.000000,0.006240,...,0.0,6.700000,0.000000,0.000000,0.2,0.2,0.2,0.2,0.2,0.0
3,AC-AcGum-ArGal-Inulin-Pectin-Starch,1.0,0.0,0.003655,NaN,NaN,0.0,0.004083,2.009172,0.006240,...,0.0,6.647682,0.000000,8.132391,0.2,0.2,0.2,0.2,0.2,0.0
4,AC-AcGum-ArGal-Inulin-Pectin-Xylan,0.0,0.0,0.004691,0.000000,NaN,0.0,0.004057,0.000000,-0.038517,...,0.0,6.700000,0.000000,0.000000,0.2,0.2,0.2,0.2,0.0,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,PC-PJ-Inulin,3.0,0.0,0.000000,0.022342,0.000923,0.0,0.001992,6.505176,0.231296,...,0.0,5.969543,1.013322,11.889233,0.0,0.0,1.0,0.0,0.0,0.0
240,PJ-Pectin-Xylan,0.0,0.0,0.011316,0.000000,-0.212745,0.0,0.005015,0.000000,-0.044790,...,0.0,6.700000,0.000000,0.000000,0.0,0.0,0.0,0.5,0.0,0.5
241,PJ-Pectin-Xylan,1.0,0.0,0.011316,0.000000,-0.212745,0.0,0.005015,1.515168,-0.044790,...,0.0,6.591297,1.379595,8.907829,0.0,0.0,0.0,0.5,0.0,0.5
242,PJ-Pectin-Xylan,2.0,0.0,0.011316,NaN,-0.212745,0.0,0.005015,NaN,-0.044790,...,0.0,6.685699,NaN,NaN,0.0,0.0,0.0,0.5,0.0,0.5


In [7]:
# full factorial of both species and controls 
n_species = len(species)
n_fibers  = len(controls)

# create matrix of all possible communities
Slist = [np.reshape(np.array(i), (1, n_species)) for i in itertools.product([0, 1], repeat = n_species)]
# remove all zeros community
S = np.array(np.concatenate(Slist)[1:], float)

# create matrix of all possible fiber combinations
Flist = [np.reshape(np.array(i), (1, n_fibers)) for i in itertools.product([0, 1], repeat = n_fibers)]
F = np.array(np.concatenate(Flist), float)

# matrix of species and fiber indeces 
M = np.array(np.zeros([S.shape[0]*F.shape[0], 2]), int)
k = 0 
for i in range(S.shape[0]):
    for j in range(F.shape[0]):
        M[k, 0] = int(i)
        M[k, 1] = int(j)
        k += 1

# function to pull sample data 
def gen_exp_cond(k):
    s_ind, f_ind = M[k]
    return S[s_ind], F[f_ind]

# function to generate informative name of experimental condition
def gen_exp_name(Si, Fi):
    exp_name = ""
    for i,si in enumerate(Si):
        if si > 0:
            exp_name += species[i].split("abs")[0] + "-"
    for i,fi in enumerate(Fi):
        if fi > 0:
            exp_name += controls[i] + "-"
    return exp_name[:-1]

In [8]:
def format_design_data(S, F, t_eval, scaler, batch_size=512):

    # data is a list of tuples (T, X, U, Y, names) where each tuple corresponds to a batch
    data = []
    
    # total number of samples
    n_samples = S.shape[0] * F.shape[0]
    k = 0 
    
    # number of evaluation times
    n_eval = len(t_eval)

    # divide data into batches
    for batch_inds in tqdm(np.array_split(np.arange(n_samples), np.ceil(n_samples / batch_size))):

        # initialize data matrices 
        T = np.empty([len(batch_inds), n_eval])
        T[:] = t_eval
        X = np.empty([len(batch_inds), S.shape[-1]+len(metabolites)])
        X[:] = np.nan
        U = np.empty([len(batch_inds), n_eval, F.shape[-1]])
        U[:] = np.nan

        # keep track of experiment names
        names = []
        for i, batch_ind in enumerate(batch_inds):

            # pull sample data 
            Si, Fi = gen_exp_cond(k)

            # keep track of experiment names
            names.append(gen_exp_name(Si, Fi))

            # store initial condition data
            X[i, :len(Si)] = .000667 * Si
            
            # initial pH and metabolites
            X[i, len(Si):] = np.array([6.7, 0., 0., 0.])
            
            # scale observed variables 
            X[i] = (X[i] - scaler.scale_dict_obs["0 min"]) / scaler.scale_dict_obs["0 max"]
            
            # store controls (already scaled)
            if sum(Fi)>0:
                U[i] =  Fi / sum(Fi)
            else:
                U[i] = Fi
            
            # update counter
            k += 1

        data.append((T, X, U, names))

    return data

In [9]:
# format and scale data based on training data
design_data = format_design_data(S, F, np.array([0, 1, 2, 3]), scaler, batch_size=1024)

100%|███████████████████████████████████████| 2048/2048 [00:41<00:00, 49.30it/s]


In [10]:
# save predictions to dataframe
pred_dfs = []
# for T, X, U, Y, exp_names in data_scaled: 
for T, X, U, exp_names in design_data:
    
    # keep initial condition
    preds = np.array(brnn.predict_point(X, U))
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        preds[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        preds[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = preds.shape
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(preds, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    pred_df[species+metabolites] = Y
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)

In [11]:
# all predicted variables
y = pred_df[observed].values

# reshape into [n_samples, n_passages, n_variables]
y_batch = np.reshape(y, [y.shape[0]//4, 4, y.shape[-1]])

# evaluate obective of every sample
obj_batch = objective_batch(y_batch)

# store objective for all time points
pred_df['Objective'] = np.repeat(obj_batch, 4)

In [11]:
# save to csv
# pred_df.to_csv("space/space.csv", index=False)